<a href="https://colab.research.google.com/github/YoussefAli10/MachineLearning/blob/main/Copy_of_Iron_deficiency_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests
from io import BytesIO

In [3]:
def load_xpt(url):
    response = requests.get(url)
    response.raise_for_status()
    return pd.read_sas(BytesIO(response.content), format="xport")

In [4]:
demo = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.XPT"
)

ferritin = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_FERTIN.XPT"
)

cbc = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_CBC.XPT"
)

diet = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DR1TOT.XPT"
)

In [5]:
print(demo.shape)
print(ferritin.shape)
print(cbc.shape)
print(diet.shape)

(15560, 29)
(11983, 3)
(13772, 22)
(14300, 168)


In [6]:
demo.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,DMDBORN4,...,FIAINTRP,MIALANG,MIAPROXY,MIAINTRP,AIALANGA,WTINTPRP,WTMECPRP,SDMVPSU,SDMVSTRA,INDFMPIR
0,109263.0,66.0,2.0,1.0,2.0,NaN,5.0,6.0,2.0,1.0,...,2.0,NaN,NaN,NaN,NaN,7891.762435,8.951816e+03,3.0,156.0,4.66
1,109264.0,66.0,2.0,2.0,13.0,NaN,1.0,1.0,2.0,1.0,...,2.0,1.0,2.0,2.0,1.0,11689.747264,1.227116e+04,1.0,155.0,0.83
2,109265.0,66.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,1.0,...,2.0,NaN,NaN,NaN,NaN,16273.825939,1.665876e+04,1.0,157.0,3.06
3,109266.0,66.0,2.0,2.0,29.0,NaN,5.0,6.0,2.0,2.0,...,2.0,1.0,2.0,2.0,1.0,7825.646112,8.154968e+03,2.0,168.0,5.00
4,109267.0,66.0,1.0,2.0,21.0,NaN,2.0,2.0,NaN,2.0,...,2.0,NaN,NaN,NaN,NaN,26379.991724,5.397605e-79,1.0,156.0,5.00


In [8]:
df = (
    demo
    .merge(ferritin, on="SEQN", how="inner")
    .merge(cbc, on="SEQN", how="inner")
    .merge(diet, on="SEQN", how="inner")
)

print(df.shape)

(11983, 219)


In [9]:
df_women = df[
    (df["RIAGENDR"] == 2) &
    (df["RIDAGEYR"] >= 20) &
    (df["RIDAGEYR"] <= 44)
].copy()

print(df_women.shape)

(1750, 219)


In [10]:
df_women["RIDEXPRG"].value_counts(dropna=False)

,count
RIDEXPRG,
2.0,1604
1.0,87
3.0,59


In [11]:
df_nonpreg = df_women[
    df_women["RIDEXPRG"] == 2
].copy()

print(df_nonpreg.shape)

(1604, 219)


In [12]:
print("Missing hemoglobin:", df_nonpreg["LBXHGB"].isna().sum())
print("Hemoglobin < 12.0:", (df_nonpreg["LBXHGB"] < 12.0).sum())

df_nonpreg["LBXHGB"].describe()

Missing hemoglobin: 61
Hemoglobin < 12.0: 239


,LBXHGB
count,1543.000000
mean,13.049968
std,1.325417
min,6.300000
25%,12.400000
50%,13.200000
75%,13.900000
max,17.200000


In [13]:
df_nonanemic = df_nonpreg[
    (df_nonpreg["LBXHGB"].notna()) &
    (df_nonpreg["LBXHGB"] >= 12.0)
].copy()

print(df_nonanemic.shape)

(1304, 219)


In [14]:
df_nonanemic["iron_deficient"] = (
    df_nonanemic["LBXFER"] < 15
).astype(int)

print(df_nonanemic["iron_deficient"].value_counts())
print(df_nonanemic["iron_deficient"].value_counts(normalize=True))

iron_deficient
0    1190
1     114
Name: count, dtype: int64
iron_deficient
0    0.912577
1    0.087423
Name: proportion, dtype: float64


In [15]:
print("Missing ferritin:", df_nonanemic["LBXFER"].isna().sum())

Missing ferritin: 23


In [16]:
df_1720 = df_nonanemic[
    df_nonanemic["LBXFER"].notna()
].copy()

df_1720["iron_deficient"] = (df_1720["LBXFER"] < 15).astype(int)

print(df_1720.shape)
print(df_1720["iron_deficient"].value_counts())

(1281, 220)
iron_deficient
0    1167
1     114
Name: count, dtype: int64


In [17]:
demo15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.XPT"
)

ferritin15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/FERTIN_I.XPT"
)

cbc15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/CBC_I.XPT"
)

diet15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DR1TOT_I.XPT"
)

print(demo15.shape, ferritin15.shape, cbc15.shape, diet15.shape)

(9971, 47) (3252, 3) (9165, 21) (9544, 168)


In [18]:
df15 = (
    demo15
    .merge(ferritin15, on="SEQN", how="inner")
    .merge(cbc15, on="SEQN", how="inner")
    .merge(diet15, on="SEQN", how="inner")
)

print(df15.shape)

(3252, 236)


In [19]:
df15_women = df15[
    (df15["RIAGENDR"] == 2) &
    (df15["RIDAGEYR"] >= 20) &
    (df15["RIDAGEYR"] <= 44)
].copy()

print(df15_women.shape)

(1240, 236)


In [20]:
df15_nonpreg = df15_women[
    df15_women["RIDEXPRG"] == 2
].copy()

print(df15_nonpreg.shape)

(1125, 236)


In [21]:
df15_nonanemic = df15_nonpreg[
    (df15_nonpreg["LBXHGB"].notna()) &
    (df15_nonpreg["LBXHGB"] >= 12.0)
].copy()

print(df15_nonanemic.shape)

(891, 236)


In [22]:
df15_clean = df15_nonanemic[
    df15_nonanemic["LBXFER"].notna()
].copy()

df15_clean["iron_deficient"] = (
    df15_clean["LBXFER"] < 15
).astype(int)

print(df15_clean.shape)
print(df15_clean["iron_deficient"].value_counts())
print(df15_clean["iron_deficient"].value_counts(normalize=True))

(888, 237)
iron_deficient
0    809
1     79
Name: count, dtype: int64
iron_deficient
0    0.911036
1    0.088964
Name: proportion, dtype: float64


In [23]:
candidate_vars = [
    "SEQN",
    "RIDAGEYR",
    "DR1TIRON",
    "DR1TVC",
    "DR1TKCAL",
    "DR1TPROT",
    "DR1TFIBE",
    "DR1TCALC",
    "DR1TCAFF",
    "DR1TALCO",
    "iron_deficient"
]

for var in candidate_vars:
    print(
        var,
        "2015-16:", var in df15_clean.columns,
        "2017-20:", var in df_1720.columns
    )

SEQN 2015-16: True 2017-20: True
RIDAGEYR 2015-16: True 2017-20: True
DR1TIRON 2015-16: True 2017-20: True
DR1TVC 2015-16: True 2017-20: True
DR1TKCAL 2015-16: True 2017-20: True
DR1TPROT 2015-16: True 2017-20: True
DR1TFIBE 2015-16: True 2017-20: True
DR1TCALC 2015-16: True 2017-20: True
DR1TCAFF 2015-16: True 2017-20: True
DR1TALCO 2015-16: True 2017-20: True
iron_deficient 2015-16: True 2017-20: True


In [24]:
train15 = df15_clean[candidate_vars].copy()
train1720 = df_1720[candidate_vars].copy()

train15["cycle"] = "2015-2016"
train1720["cycle"] = "2017-2020"

In [25]:
train_combined = pd.concat(
    [train15, train1720],
    ignore_index=True
)

print(train_combined.shape)
print(train_combined["iron_deficient"].value_counts())

(2169, 12)
iron_deficient
0    1976
1     193
Name: count, dtype: int64


In [26]:
rhq15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/RHQ_I.XPT"
)

paq15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/PAQ_I.XPT"
)

smq15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/SMQ_I.XPT"
)

alq15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/ALQ_I.XPT"
)

supp15 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DSQTOT_I.XPT"
)

In [27]:
rhq1720 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_RHQ.XPT"
)

paq1720 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PAQ.XPT"
)

smq1720 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.XPT"
)

alq1720 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_ALQ.XPT"
)

supp1720 = load_xpt(
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DSQTOT.XPT"
)

In [28]:
print("2015-16:")
print(rhq15.shape, paq15.shape, smq15.shape, alq15.shape, supp15.shape)

print("2017-20:")
print(rhq1720.shape, paq1720.shape, smq1720.shape, alq1720.shape, supp1720.shape)

2015-16:
(3471, 46) (9255, 94) (7001, 42) (5735, 10) (9971, 39)
2017-20:
(5314, 32) (9693, 17) (11137, 16) (8965, 10) (15560, 39)


In [29]:
candidate_extra_vars = {
    "reproductive": [
        "RHQ031",   # period in past 12 months
        "RHD043",   # reason for no periods
        "RHQ200",   # breastfeeding
        "RHD280"    # hysterectomy
    ],

    "physical_activity": [
        "PAQ650",   # vigorous recreation
        "PAQ665",   # moderate recreation
        "PAD680"    # sedentary minutes/day
    ],

    "smoking": [
        "SMQ020",   # smoked 100 cigarettes lifetime
        "SMQ040"    # current smoking status
    ],

    "alcohol": [
        "ALQ111",
        "ALQ121",
        "ALQ130",
        "ALQ170"
    ],

    "supplements": [
        "DSD010",    # used supplements
        "DSDCOUNT",  # number of supplements
        "DSQTIRON",  # supplemental iron
        "DSQTVC",    # supplemental vitamin C
        "DSQTCALC"   # supplemental calcium
    ]
}

pairs = {
    "reproductive": (rhq15, rhq1720),
    "physical_activity": (paq15, paq1720),
    "smoking": (smq15, smq1720),
    "alcohol": (alq15, alq1720),
    "supplements": (supp15, supp1720)
}

for group, variables in candidate_extra_vars.items():
    print(f"\n{group.upper()}")

    df_old, df_new = pairs[group]

    for var in variables:
        print(
            var,
            "2015-16:", var in df_old.columns,
            "2017-20:", var in df_new.columns
        )


REPRODUCTIVE
RHQ031 2015-16: True 2017-20: True
RHD043 2015-16: True 2017-20: True
RHQ200 2015-16: True 2017-20: True
RHD280 2015-16: True 2017-20: True

PHYSICAL_ACTIVITY
PAQ650 2015-16: True 2017-20: True
PAQ665 2015-16: True 2017-20: True
PAD680 2015-16: True 2017-20: True

SMOKING
SMQ020 2015-16: True 2017-20: True
SMQ040 2015-16: True 2017-20: True

ALCOHOL
ALQ111 2015-16: False 2017-20: True
ALQ121 2015-16: False 2017-20: True
ALQ130 2015-16: True 2017-20: True
ALQ170 2015-16: False 2017-20: True

SUPPLEMENTS
DSD010 2015-16: True 2017-20: True
DSDCOUNT 2015-16: True 2017-20: True
DSQTIRON 2015-16: True 2017-20: True
DSQTVC 2015-16: True 2017-20: True
DSQTCALC 2015-16: True 2017-20: True


In [30]:
extra_vars = [
    # Reproductive
    "RHQ031",
    "RHQ200",
    "RHD280",

    # Physical activity
    "PAQ650",
    "PAQ665",
    "PAD680",

    # Smoking
    "SMQ020",
    "SMQ040",

    # Alcohol
    "ALQ130",

    # Supplements
    "DSD010",
    "DSDCOUNT",
    "DSQTIRON",
    "DSQTVC",
    "DSQTCALC"
]

In [31]:
extra15 = (
    rhq15[["SEQN", "RHQ031", "RHQ200", "RHD280"]]
    .merge(paq15[["SEQN", "PAQ650", "PAQ665", "PAD680"]], on="SEQN", how="outer")
    .merge(smq15[["SEQN", "SMQ020", "SMQ040"]], on="SEQN", how="outer")
    .merge(alq15[["SEQN", "ALQ130"]], on="SEQN", how="outer")
    .merge(
        supp15[["SEQN", "DSD010", "DSDCOUNT", "DSQTIRON", "DSQTVC", "DSQTCALC"]],
        on="SEQN",
        how="outer"
    )
)

train15_full = train15.merge(extra15, on="SEQN", how="left")

In [32]:
extra1720 = (
    rhq1720[["SEQN", "RHQ031", "RHQ200", "RHD280"]]
    .merge(paq1720[["SEQN", "PAQ650", "PAQ665", "PAD680"]], on="SEQN", how="outer")
    .merge(smq1720[["SEQN", "SMQ020", "SMQ040"]], on="SEQN", how="outer")
    .merge(alq1720[["SEQN", "ALQ130"]], on="SEQN", how="outer")
    .merge(
        supp1720[["SEQN", "DSD010", "DSDCOUNT", "DSQTIRON", "DSQTVC", "DSQTCALC"]],
        on="SEQN",
        how="outer"
    )
)

train1720_full = train1720.merge(extra1720, on="SEQN", how="left")

In [33]:
train_combined = pd.concat(
    [train15_full, train1720_full],
    ignore_index=True
)

print(train_combined.shape)

(2169, 26)


In [34]:
missing = train_combined.isna().sum().sort_values(ascending=False)

print(missing)

RHQ200            1884
DSQTIRON          1776
SMQ040            1547
DSQTCALC          1465
DSQTVC            1462
ALQ130             615
RHD280             243
RHQ031             225
DR1TVC             186
DR1TIRON           186
DR1TPROT           186
DR1TKCAL           186
DR1TALCO           186
DR1TCAFF           186
DR1TCALC           186
DR1TFIBE           186
PAD680               2
RIDAGEYR             0
SEQN                 0
PAQ650               0
cycle                0
iron_deficient       0
PAQ665               0
SMQ020               0
DSD010               0
DSDCOUNT             0
dtype: int64


In [35]:
missing_pct = (
    train_combined.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(missing_pct)

RHQ200            86.860304
DSQTIRON          81.881051
SMQ040            71.323190
DSQTCALC          67.542646
DSQTVC            67.404334
ALQ130            28.354080
RHD280            11.203320
RHQ031            10.373444
DR1TVC             8.575380
DR1TIRON           8.575380
DR1TPROT           8.575380
DR1TKCAL           8.575380
DR1TALCO           8.575380
DR1TCAFF           8.575380
DR1TCALC           8.575380
DR1TFIBE           8.575380
PAD680             0.092208
RIDAGEYR           0.000000
SEQN               0.000000
PAQ650             0.000000
cycle              0.000000
iron_deficient     0.000000
PAQ665             0.000000
SMQ020             0.000000
DSD010             0.000000
DSDCOUNT           0.000000
dtype: float64


In [36]:
# Drop variables we do not want in the first baseline
train_combined = train_combined.drop(
    columns=["RHQ200", "ALQ130"]
)

# Nutrient intake from supplements:
# no reported amount -> 0 mg/day
for col in ["DSQTIRON", "DSQTVC", "DSQTCALC"]:
    train_combined[col] = train_combined[col].fillna(0)

In [37]:
import numpy as np

def smoking_status(row):
    # Never smoked 100 cigarettes
    if row["SMQ020"] == 2:
        return "never"

    # Smoked 100+ cigarettes and currently smokes
    if row["SMQ020"] == 1 and row["SMQ040"] in [1, 2]:
        return "current"

    # Smoked 100+ cigarettes but not currently
    if row["SMQ020"] == 1 and row["SMQ040"] == 3:
        return "former"

    return np.nan

train_combined["smoking_status"] = train_combined.apply(
    smoking_status,
    axis=1
)

train_combined = train_combined.drop(
    columns=["SMQ020", "SMQ040"]
)

In [38]:
(train_combined.isna().mean() * 100).sort_values(ascending=False)

,0
RHD280,11.203320
RHQ031,10.373444
DR1TIRON,8.575380
DR1TPROT,8.575380
DR1TFIBE,8.575380
DR1TKCAL,8.575380
DR1TVC,8.575380
DR1TALCO,8.575380
DR1TCAFF,8.575380
DR1TCALC,8.575380


In [39]:
categorical_check = [
    "RHD280",
    "RHQ031",
    "PAQ650",
    "PAQ665",
    "DSD010",
    "smoking_status"
]

for col in categorical_check:
    print(f"\n{col}")
    print(train_combined[col].value_counts(dropna=False).sort_index())


RHD280
RHD280
1.0      86
2.0    1840
NaN     243
Name: count, dtype: int64

RHQ031
RHQ031
1.0    1728
2.0     216
NaN     225
Name: count, dtype: int64

PAQ650
PAQ650
1.0     726
2.0    1443
Name: count, dtype: int64

PAQ665
PAQ665
1.0    1041
2.0    1128
Name: count, dtype: int64

DSD010
DSD010
1.0    1054
2.0    1115
Name: count, dtype: int64

smoking_status
smoking_status
current     372
former      250
never      1545
NaN           2
Name: count, dtype: int64


In [40]:
numeric_features = [
    "RIDAGEYR",
    "DR1TIRON",
    "DR1TVC",
    "DR1TKCAL",
    "DR1TPROT",
    "DR1TFIBE",
    "DR1TCALC",
    "DR1TCAFF",
    "DR1TALCO",
    "PAD680",
    "DSDCOUNT",
    "DSQTIRON",
    "DSQTVC",
    "DSQTCALC"
]

categorical_features = [
    "RHD280",
    "RHQ031",
    "PAQ650",
    "PAQ665",
    "DSD010",
    "smoking_status"
]

X = train_combined[numeric_features + categorical_features].copy()
y = train_combined["iron_deficient"].copy()

print(X.shape)
print(y.value_counts())

(2169, 20)
iron_deficient
0    1976
1     193
Name: count, dtype: int64


In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", y_train.value_counts())
print("Testing:", y_test.value_counts())

Training: iron_deficient
0    1581
1     154
Name: count, dtype: int64
Testing: iron_deficient
0    395
1     39
Name: count, dtype: int64


In [43]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

dummy_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

dummy_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['RIDAGEYR', 'DR1TIRON',
                                                   'DR1TVC', 'DR1TKCAL',
                                                   'DR1TPROT', 'DR1TFIBE',
                                                   'DR1TCALC', 'DR1TCAFF',
                                                   'DR1TALCO', 'PAD680',
                                                   'DSDCOUNT', 'DSQTIRON',
                                                   'DSQTVC', 'DSQTCALC']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['RHD280', 'RHQ031', 'PAQ650',
                                                   'PAQ665', 'DSD010',
                                                   'smoking_status'])])),
                ('classifier', DummyClassifier(strategy='most_frequent'))])

In [44]:
y_pred_dummy = dummy_model.predict(X_test)

In [45]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Precision:", precision_score(y_test, y_pred_dummy, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_dummy, zero_division=0))
print("F1:", f1_score(y_test, y_pred_dummy, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dummy))

Accuracy: 0.9101382488479263
Precision: 0.0
Recall: 0.0
F1: 0.0

Confusion Matrix:
[[395   0]
 [ 39   0]]


In [46]:
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

logreg_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['RIDAGEYR', 'DR1TIRON',
                                                   'DR1TVC', 'DR1TKCAL',
                                                   'DR1TPROT', 'DR1TFIBE',
                                                   'DR1TCALC', 'DR1TCAFF',
                                                   'DR1TALCO', 'PAD680',
                                                   'DSDCOUNT', 'DSQTIRON',
                                                   'DSQTVC', 'DSQTCALC']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['RHD280', 'RHQ031', 'PAQ650',
                                                   'PAQ665', 'DSD010',
                                                   'smoking_status'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [47]:
y_pred_logreg = logreg_model.predict(X_test)
y_prob_logreg = logreg_model.predict_proba(X_test)[:, 1]

In [48]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred_logreg))
print("Precision:", precision_score(y_test, y_pred_logreg))
print("Recall:", recall_score(y_test, y_pred_logreg))
print("F1:", f1_score(y_test, y_pred_logreg))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_logreg))
print("PR-AUC:", average_precision_score(y_test, y_prob_logreg))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logreg))

Accuracy: 0.4815668202764977
Precision: 0.09565217391304348
Recall: 0.5641025641025641
F1: 0.16356877323420074
ROC-AUC: 0.5289191820837391
PR-AUC: 0.09303422319615089

Confusion Matrix:
[[187 208]
 [ 17  22]]
